In [ ]:
# =====================================================================
# IEEE ACCESS R1 — PHASE 4
# REPEATED-TRAINING-SEED VARIABILITY
#
# PURPOSE
# -------
# Address Reviewer 1 Comment 5:
#
# Existing bootstrap intervals quantify variation over test observations
# but do not quantify neural-training stochasticity because each fold was
# trained with one fixed random seed.
#
# THIS EXPERIMENT USES FIVE COMPLETE SEED SCHEDULES:
#
# S0  = frozen original submitted schedule
#       folds: 142, 242, 342, 442, 542
#
# S1  = 1142, 1242, 1342, 1442, 1542
# S2  = 2142, 2242, 2342, 2442, 2542
# S3  = 3142, 3242, 3342, 3442, 3542
# S4  = 4142, 4242, 4342, 4442, 4542
#
# S0 is reused from the exact frozen original experiment.
# S1-S4 are newly trained.
#
# ELITE:
#   A-MLP
#   C-Hybrid-positional
#   C-Hybrid-globalmean
#
# GLOBAL:
#   A-MLP
#   C-Hybrid-positional
#
# QUESTIONS
# ---------
# 1. Is C-Hybrid vs A-MLP stable across training seeds?
# 2. Is the Elite positional-pooling advantage stable across seeds?
#
# IMPORTANT
# ---------
# - Same frozen folds.
# - Same features.
# - Same training-only preprocessing.
# - Same optimizer/hyperparameters.
# - Same early-stopping protocol.
# - Early-stopping epoch is RESELECTED for every new seed.
# - Same OOF test observations.
# - No seed is selected because it produces a preferred result.
#
# This analysis is DESCRIPTIVE training-stability analysis.
# We report mean, sample SD, min, max and direction consistency across
# five complete seed schedules.
# =====================================================================


# =====================================================================
# 0. IMPORTS
# =====================================================================

import os
import sys
import gc
import json
import time
import shutil
import hashlib
import zipfile
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd


# =====================================================================
# 1. FROZEN EXPERIMENT SETTINGS
# =====================================================================

REPO_URL = (
    "https://github.com/"
    "laknathvg/outcome-linked-football-passing-gnn.git"
)

REPO_DIR = Path(
    "/content/outcome-linked-football-passing-gnn"
)

# Same frozen source used for the earlier revision experiments.
FROZEN_COMMIT = (
    "e5fdb3ceafa08a82789264a8d57fe8f51a8daea1"
)

WORK_ROOT = Path(
    "/content/IEEE_Access_Phase4_Work"
)

CHECKPOINT_DIR = (
    WORK_ROOT
    / "checkpoints"
)

RESULT_DIR = (
    WORK_ROOT
    / "outputs"
)

FINAL_ZIP = Path(
    "/content/"
    "IEEE_Access_R1_Phase4_Repeated_Seed_Variability.zip"
)

PROTOCOL_ID = (
    "phase4_v1_"
    "5_seed_schedules_"
    "S0_frozen_S1_S4_new_"
    "elite_main_and_pooling_"
    "global_main"
)

BASE_SEED = 42

# Four NEW schedules plus frozen original S0.
SEED_SCHEDULES = {
    "S0": 0,
    "S1": 1000,
    "S2": 2000,
    "S3": 3000,
    "S4": 4000,
}

NEW_SCHEDULES = [
    "S1",
    "S2",
    "S3",
    "S4",
]

EXPECTED = {
    "elite": {
        "eligible_n": 757,
        "oof_n": 454,
        "models": [
            "A-MLP",
            "C-Hybrid-positional",
            "C-Hybrid-globalmean",
        ],
    },

    "global": {
        "eligible_n": 3098,
        "oof_n": 1745,
        "models": [
            "A-MLP",
            "C-Hybrid-positional",
        ],
    },
}


print("=" * 105)
print("IEEE ACCESS R1 — PHASE 4")
print("REPEATED-TRAINING-SEED VARIABILITY")
print("=" * 105)


# =====================================================================
# 2. GPU CHECK
# =====================================================================

import torch

print("\nPyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0),
    )

else:

    raise RuntimeError(
        "CUDA GPU is not enabled.\n\n"
        "In Colab choose:\n"
        "Runtime -> Change runtime type -> T4 GPU\n"
        "then rerun this cell."
    )


device = torch.device(
    "cuda"
)


# =====================================================================
# 3. CREATE / VERIFY RESUMABLE WORK DIRECTORY
# =====================================================================

WORK_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


protocol_marker = (
    WORK_ROOT
    / "phase4_protocol_marker.json"
)


expected_marker = {
    "protocol_id":
        PROTOCOL_ID,

    "frozen_commit":
        FROZEN_COMMIT,

    "seed_schedules":
        SEED_SCHEDULES,

    "new_schedules":
        NEW_SCHEDULES,

    "elite_models":
        EXPECTED[
            "elite"
        ][
            "models"
        ],

    "global_models":
        EXPECTED[
            "global"
        ][
            "models"
        ],
}


if protocol_marker.exists():

    existing_marker = json.loads(
        protocol_marker.read_text(
            encoding="utf-8"
        )
    )

    if existing_marker != expected_marker:

        raise RuntimeError(
            "Existing Phase 4 checkpoint directory was created "
            "with a different protocol.\n\n"
            "Do NOT mix experiments.\n"
            "Delete /content/IEEE_Access_Phase4_Work only if "
            "you intentionally want to restart from zero."
        )

    print(
        "\n✓ Existing compatible Phase 4 checkpoints detected."
    )

    print(
        "Completed fold/seed combinations will be reused."
    )

else:

    protocol_marker.write_text(
        json.dumps(
            expected_marker,
            indent=2,
        ),
        encoding="utf-8",
    )

    print(
        "\n✓ New frozen Phase 4 protocol created."
    )


# =====================================================================
# 4. CLONE EXACT FROZEN REPOSITORY SOURCE
# =====================================================================

if REPO_DIR.exists():

    shutil.rmtree(
        REPO_DIR
    )


print("\nCloning frozen research repository...")


subprocess.run(
    [
        "git",
        "clone",
        REPO_URL,
        str(REPO_DIR),
    ],
    check=True,
)


os.chdir(
    REPO_DIR
)


subprocess.run(
    [
        "git",
        "checkout",
        "--detach",
        FROZEN_COMMIT,
    ],
    check=True,
)


actual_commit = subprocess.check_output(
    [
        "git",
        "rev-parse",
        "HEAD",
    ],
    text=True,
).strip()


if actual_commit != FROZEN_COMMIT:

    raise RuntimeError(
        "Repository commit mismatch.\n"
        f"Expected: {FROZEN_COMMIT}\n"
        f"Actual:   {actual_commit}"
    )


print(
    "✓ Frozen repository commit verified:"
)

print(
    actual_commit
)


# =====================================================================
# 5. INSTALL EXACT REPOSITORY REQUIREMENTS
# =====================================================================

print(
    "\nInstalling repository requirements..."
)


subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        "requirements.txt",
    ],
    check=True,
)


print(
    "✓ Requirements installed."
)


# =====================================================================
# 6. IMPORT RESEARCH PIPELINE
# =====================================================================

import yaml

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from sklearn.preprocessing import (
    StandardScaler,
)

from src.evaluation import (
    prediction_rows,
)

from src.final_protocol import (
    harvest_dataset_with_retry_cache,
)

from src.models import (
    HistoryMLP,
    HybridGNN,
)

from src.reproducibility import (
    save_environment,
)

from src.scaling import (
    fit_graph_scalers,
    make_feature_view,
    transform_graphs,
)

from src.training import (
    select_history_epoch,
    fit_history_model,
    predict_history_model,
    select_graph_epoch,
    fit_graph_model,
    predict_graph_model,
)


# =====================================================================
# 7. FROZEN SEED SCHEDULE
# =====================================================================

def seed_for(
    schedule_id,
    fold,
):

    offset = SEED_SCHEDULES[
        schedule_id
    ]

    return (
        BASE_SEED
        + 100 * int(fold)
        + int(offset)
    )


seed_rows = []


for schedule_id, offset in SEED_SCHEDULES.items():

    for fold in range(
        1,
        6,
    ):

        seed_rows.append({
            "schedule_id":
                schedule_id,

            "schedule_index":
                int(
                    schedule_id[
                        1:
                    ]
                ),

            "offset":
                offset,

            "fold":
                fold,

            "training_seed":
                seed_for(
                    schedule_id,
                    fold,
                ),

            "source":
                (
                    "frozen_original"
                    if schedule_id == "S0"
                    else "new_retraining"
                ),
        })


seed_schedule_df = pd.DataFrame(
    seed_rows
)


seed_schedule_df.to_csv(
    RESULT_DIR
    / "seed_schedule.csv",
    index=False,
)


print("\nFrozen seed schedules:")

print(
    seed_schedule_df
    .pivot(
        index="schedule_id",
        columns="fold",
        values="training_seed",
    )
    .to_string()
)


# =====================================================================
# 8. LOAD CONFIGURATION
# =====================================================================

def load_config(
    cohort,
):

    if cohort == "elite":

        config_path = (
            REPO_DIR
            / "config_final.yaml"
        )

    elif cohort == "global":

        config_path = (
            REPO_DIR
            / "config_experiment5_global.yaml"
        )

    else:

        raise ValueError(
            cohort
        )


    with config_path.open(
        "r",
        encoding="utf-8",
    ) as handle:

        config = yaml.safe_load(
            handle
        )


    if cohort == "elite":

        config[
            "project"
        ][
            "cache_dir"
        ] = (
            "/content/"
            "statsbomb_phase4_elite_cache"
        )

    else:

        config[
            "project"
        ][
            "cache_dir"
        ] = (
            "/content/"
            "statsbomb_global_event_cache"
        )


    # Frozen architecture/training audit.
    assert int(
        config[
            "graph"
        ][
            "node_count"
        ]
    ) == 11

    assert int(
        config[
            "model"
        ][
            "hidden_dim"
        ]
    ) == 64

    assert int(
        config[
            "model"
        ][
            "attention_heads"
        ]
    ) == 4

    assert bool(
        config[
            "model"
        ][
            "attention_concat"
        ]
    ) is False

    assert int(
        config[
            "model"
        ][
            "role_embedding_dim"
        ]
    ) == 8

    assert float(
        config[
            "model"
        ][
            "graph_dropout"
        ]
    ) == 0.30

    assert float(
        config[
            "model"
        ][
            "classifier_dropout"
        ]
    ) == 0.40

    assert int(
        config[
            "model"
        ][
            "history_hidden_dim"
        ]
    ) == 32

    assert float(
        config[
            "model"
        ][
            "history_dropout"
        ]
    ) == 0.20

    assert float(
        config[
            "training"
        ][
            "learning_rate"
        ]
    ) == 0.002

    assert float(
        config[
            "training"
        ][
            "weight_decay"
        ]
    ) == 0.0005

    assert int(
        config[
            "training"
        ][
            "batch_size"
        ]
    ) == 32

    assert int(
        config[
            "training"
        ][
            "maximum_epochs"
        ]
    ) == 50

    assert int(
        config[
            "training"
        ][
            "early_stopping"
        ][
            "patience"
        ]
    ) == 10

    return config


# =====================================================================
# 9. LOAD FROZEN COHORT FILES
# =====================================================================

def load_frozen_cohort(
    cohort,
):

    if cohort == "elite":

        samples_path = (
            REPO_DIR
            / "reference_experiment1"
            / "included_samples.csv"
        )

        history_path = (
            REPO_DIR
            / "reference_experiment1"
            / "history_feature_order.json"
        )

        assignment_path = (
            REPO_DIR
            / "reference_experiment1"
            / "fold_assignments.csv"
        )

    else:

        base = (
            REPO_DIR
            / "outputs"
            / "experiment5_global_pooled_consistency"
        )

        samples_path = (
            base
            / "included_samples.csv"
        )

        history_path = (
            base
            / "history_feature_order.json"
        )

        assignment_path = (
            base
            / "global_fold_assignments.csv"
        )


    samples = pd.read_csv(
        samples_path
    ).reset_index(
        drop=True
    )


    with history_path.open(
        "r",
        encoding="utf-8",
    ) as handle:

        history_columns = json.load(
            handle
        )


    assignments = pd.read_csv(
        assignment_path
    )


    expected_n = EXPECTED[
        cohort
    ][
        "eligible_n"
    ]


    if len(
        samples
    ) != expected_n:

        raise RuntimeError(
            f"{cohort}: frozen eligible N mismatch.\n"
            f"Expected {expected_n}, got {len(samples)}"
        )


    if len(
        history_columns
    ) != 16:

        raise RuntimeError(
            f"{cohort}: expected 16 historical features."
        )


    missing_history = [
        column
        for column in history_columns
        if column not in samples.columns
    ]


    if missing_history:

        raise RuntimeError(
            f"{cohort}: frozen history features missing:\n"
            f"{missing_history}"
        )


    required_assignment_columns = {
        "fold",
        "level",
        "role",
        "match_id",
    }


    missing_assignment_columns = (
        required_assignment_columns
        - set(
            assignments.columns
        )
    )


    if missing_assignment_columns:

        raise RuntimeError(
            f"{cohort}: malformed frozen fold assignments."
        )


    folds = sorted(
        assignments[
            "fold"
        ]
        .astype(
            int
        )
        .unique()
        .tolist()
    )


    if folds != [
        1,
        2,
        3,
        4,
        5,
    ]:

        raise RuntimeError(
            f"{cohort}: expected folds 1-5, got {folds}"
        )


    samples[
        "match_id"
    ] = samples[
        "match_id"
    ].astype(
        int
    )


    assignments[
        "match_id"
    ] = assignments[
        "match_id"
    ].astype(
        int
    )


    y = samples[
        "label"
    ].astype(
        int
    ).to_numpy()


    X_history = (
        samples[
            history_columns
        ]
        .astype(
            float
        )
        .to_numpy()
    )


    id_to_index = {
        int(
            match_id
        ):
            int(
                index
            )

        for index, match_id
        in enumerate(
            samples[
                "match_id"
            ].tolist()
        )
    }


    assignment_ids = set(
        assignments[
            "match_id"
        ].astype(
            int
        )
    )


    missing_assignment_ids = (
        assignment_ids
        - set(
            id_to_index.keys()
        )
    )


    if missing_assignment_ids:

        raise RuntimeError(
            f"{cohort}: fold assignments contain matches "
            f"outside frozen sample table."
        )


    return (
        samples,
        history_columns,
        assignments,
        y,
        X_history,
        id_to_index,
    )


# =====================================================================
# 10. EXACT FOLD INDICES
# =====================================================================

def indices_for(
    assignments,
    id_to_index,
    fold,
    level,
    role,
):

    ids = (
        assignments.loc[
            assignments[
                "fold"
            ].eq(
                fold
            )
            &
            assignments[
                "level"
            ].eq(
                level
            )
            &
            assignments[
                "role"
            ].eq(
                role
            ),
            "match_id",
        ]
        .astype(
            int
        )
        .tolist()
    )


    if not ids:

        raise RuntimeError(
            f"Missing partition: "
            f"fold={fold}, level={level}, role={role}"
        )


    return np.asarray(
        [
            id_to_index[
                match_id
            ]
            for match_id in ids
        ],
        dtype=int,
    )


# =====================================================================
# 11. GRAPH FEATURE HELPERS
# =====================================================================

def full_graph_views(
    graphs,
    indices,
):

    output = []

    for index in indices:

        graph = graphs[
            int(
                index
            )
        ]

        if graph is None:

            raise RuntimeError(
                f"Missing graph at frozen sample index {index}."
            )

        output.append(
            make_feature_view(
                graph,
                node_indices=[
                    0,
                    1,
                    2,
                    3,
                ],
                edge_indices=[
                    0,
                    1,
                    2,
                ],
            )
        )

    return output


def preprocess_graph_pair(
    train_raw,
    other_raw,
    train_history,
    other_history,
):

    scalers = fit_graph_scalers(
        graphs=train_raw,
        standardize_nodes=True,
        edge_continuous_indices=[
            0,
            1,
        ],
    )


    train_scaled = transform_graphs(
        train_raw,
        scalers,
        train_history,
    )


    other_scaled = transform_graphs(
        other_raw,
        scalers,
        other_history,
    )


    return (
        train_scaled,
        other_scaled,
    )


# =====================================================================
# 12. RECONSTRUCT ONLY THE FROZEN REQUIRED MATCH GRAPHS
# =====================================================================

def reconstruct_frozen_graphs(
    cohort,
    config,
    samples,
    assignments,
):

    required_ids = set(
        assignments[
            "match_id"
        ].astype(
            int
        )
    )


    print(
        f"\n{cohort.upper()} — reconstructing StatsBomb graphs..."
    )


    (
        match_stats,
        graph_map,
        manifest,
        candidate_manifest,
    ) = harvest_dataset_with_retry_cache(
        config,
        attempts=5,
        fail_on_event_error=False,
    )


    missing_required = (
        required_ids
        - set(
            graph_map.keys()
        )
    )


    if missing_required:

        print(
            f"{len(missing_required)} frozen required graphs "
            f"missing after first pass."
        )

        print(
            "Retrying unresolved event requests using cache..."
        )

        time.sleep(
            3
        )

        (
            match_stats,
            graph_map,
            manifest,
            candidate_manifest,
        ) = harvest_dataset_with_retry_cache(
            config,
            attempts=8,
            fail_on_event_error=False,
        )


        missing_required = (
            required_ids
            - set(
                graph_map.keys()
            )
        )


    if missing_required:

        raise RuntimeError(
            f"{cohort}: could not reconstruct "
            f"{len(missing_required)} required frozen graphs.\n"
            f"First IDs: {sorted(missing_required)[:20]}\n\n"
            "Do NOT change the cohort. Rerun the same cell "
            "while the event cache remains available."
        )


    # Validate labels against frozen sample table.
    sample_lookup = (
        samples
        .set_index(
            "match_id"
        )
    )


    for match_id in required_ids:

        frozen_label = int(
            sample_lookup.loc[
                match_id,
                "label",
            ]
        )

        reconstructed_label = int(
            graph_map[
                match_id
            ]
            .y
            .view(
                -1
            )[0]
            .item()
        )


        if (
            frozen_label
            != reconstructed_label
        ):

            raise RuntimeError(
                f"{cohort}: label mismatch for match {match_id}."
            )


    graphs = []


    for match_id in samples[
        "match_id"
    ].astype(
        int
    ):

        if match_id in graph_map:

            graphs.append(
                graph_map[
                    match_id
                ]
            )

        else:

            # Matches outside the frozen OOF-evaluable assignments
            # may be unused by Phase 4.
            graphs.append(
                None
            )


    print(
        f"✓ {cohort.upper()} required frozen graphs verified."
    )


    del match_stats
    del manifest
    del candidate_manifest
    del graph_map

    gc.collect()

    return graphs


# =====================================================================
# 13. FROZEN ORIGINAL S0 PREDICTIONS
# =====================================================================

def build_original_S0_predictions(
    cohort,
):

    if cohort == "elite":

        original = pd.read_csv(
            REPO_DIR
            / "reference_experiment1"
            / "fold_predictions_final.csv"
        )


        pooling = pd.read_csv(
            REPO_DIR
            / "outputs"
            / "experiment3_pooling_ablation"
            / "pooling_predictions.csv"
        )


        a_mlp = original[
            original[
                "model"
            ].eq(
                "A-MLP"
            )
        ].copy()


        hybrid_pos = original[
            original[
                "model"
            ].eq(
                "C-Hybrid-full"
            )
        ].copy()


        hybrid_pos[
            "model"
        ] = (
            "C-Hybrid-positional"
        )


        hybrid_global = pooling[
            pooling[
                "model"
            ].eq(
                "C-Hybrid-globalmean"
            )
        ].copy()


        frames = [
            a_mlp,
            hybrid_pos,
            hybrid_global,
        ]


    else:

        original = pd.read_csv(
            REPO_DIR
            / "outputs"
            / "experiment5_global_pooled_consistency"
            / "global_oof_predictions.csv"
        )


        a_mlp = original[
            original[
                "model"
            ].eq(
                "A-MLP"
            )
        ].copy()


        hybrid_pos = original[
            original[
                "model"
            ].eq(
                "C-Hybrid-full"
            )
        ].copy()


        hybrid_pos[
            "model"
        ] = (
            "C-Hybrid-positional"
        )


        frames = [
            a_mlp,
            hybrid_pos,
        ]


    output = pd.concat(
        frames,
        ignore_index=True,
        sort=False,
    )


    output[
        "seed_schedule"
    ] = "S0"

    output[
        "seed_schedule_index"
    ] = 0

    output[
        "seed_offset"
    ] = 0

    output[
        "seed_source"
    ] = "frozen_original"


    expected_n = EXPECTED[
        cohort
    ][
        "oof_n"
    ]


    expected_models = EXPECTED[
        cohort
    ][
        "models"
    ]


    for model_name in expected_models:

        model_block = output[
            output[
                "model"
            ].eq(
                model_name
            )
        ]


        if len(
            model_block
        ) != expected_n:

            raise RuntimeError(
                f"{cohort} S0 {model_name}: "
                f"expected {expected_n} predictions, "
                f"got {len(model_block)}"
            )


    match_sets = [
        set(
            output[
                output[
                    "model"
                ].eq(
                    model_name
                )
            ][
                "match_id"
            ].astype(
                int
            )
        )

        for model_name in expected_models
    ]


    if not all(
        ids == match_sets[
            0
        ]
        for ids in match_sets[
            1:
        ]
    ):

        raise RuntimeError(
            f"{cohort}: frozen S0 model predictions "
            "do not use identical test observations."
        )


    output_path = (
        CHECKPOINT_DIR
        / f"{cohort}_S0_frozen_predictions.csv"
    )


    output.to_csv(
        output_path,
        index=False,
    )


    return output


# =====================================================================
# 14. FROZEN ORIGINAL S0 SELECTED EPOCHS
# =====================================================================

def original_S0_epoch_rows(
    cohort,
):

    rows = []


    if cohort == "elite":

        elite_epochs = pd.read_csv(
            REPO_DIR
            / "reference_experiment1"
            / "selected_epochs.csv"
        )


        for record in elite_epochs.itertuples(
            index=False
        ):

            rows.append({
                "cohort":
                    "elite",

                "seed_schedule":
                    "S0",

                "seed_schedule_index":
                    0,

                "fold":
                    int(
                        record.fold
                    ),

                "seed":
                    int(
                        record.seed
                    ),

                "model":
                    "A-MLP",

                "best_epoch":
                    int(
                        record.A_MLP_best_epoch
                    ),

                "validation_macro_f1":
                    float(
                        record.A_MLP_validation_macro_f1
                    ),

                "source":
                    "frozen_original",
            })


            rows.append({
                "cohort":
                    "elite",

                "seed_schedule":
                    "S0",

                "seed_schedule_index":
                    0,

                "fold":
                    int(
                        record.fold
                    ),

                "seed":
                    int(
                        record.seed
                    ),

                "model":
                    "C-Hybrid-positional",

                "best_epoch":
                    int(
                        record.C_Hybrid_best_epoch
                    ),

                "validation_macro_f1":
                    float(
                        record.C_Hybrid_validation_macro_f1
                    ),

                "source":
                    "frozen_original",
            })


        pooling_epochs = pd.read_csv(
            REPO_DIR
            / "outputs"
            / "experiment3_pooling_ablation"
            / "pooling_selected_epochs.csv"
        )


        pooling_epochs = pooling_epochs[
            pooling_epochs[
                "model"
            ].eq(
                "C-Hybrid-globalmean"
            )
        ]


        for record in pooling_epochs.itertuples(
            index=False
        ):

            rows.append({
                "cohort":
                    "elite",

                "seed_schedule":
                    "S0",

                "seed_schedule_index":
                    0,

                "fold":
                    int(
                        record.fold
                    ),

                "seed":
                    int(
                        record.seed
                    ),

                "model":
                    "C-Hybrid-globalmean",

                "best_epoch":
                    int(
                        record.best_epoch
                    ),

                "validation_macro_f1":
                    float(
                        record.inner_validation_macro_f1
                    ),

                "source":
                    "frozen_original",
            })


    else:

        global_epochs = pd.read_csv(
            REPO_DIR
            / "outputs"
            / "experiment5_global_pooled_consistency"
            / "global_selected_epochs.csv"
        )


        for record in global_epochs.itertuples(
            index=False
        ):

            rows.append({
                "cohort":
                    "global",

                "seed_schedule":
                    "S0",

                "seed_schedule_index":
                    0,

                "fold":
                    int(
                        record.fold
                    ),

                "seed":
                    int(
                        record.seed
                    ),

                "model":
                    "A-MLP",

                "best_epoch":
                    int(
                        record.A_MLP_best_epoch
                    ),

                "validation_macro_f1":
                    float(
                        record.A_MLP_validation_macro_f1
                    ),

                "source":
                    "frozen_original",
            })


            rows.append({
                "cohort":
                    "global",

                "seed_schedule":
                    "S0",

                "seed_schedule_index":
                    0,

                "fold":
                    int(
                        record.fold
                    ),

                "seed":
                    int(
                        record.seed
                    ),

                "model":
                    "C-Hybrid-positional",

                "best_epoch":
                    int(
                        record.C_Hybrid_best_epoch
                    ),

                "validation_macro_f1":
                    float(
                        record.C_Hybrid_validation_macro_f1
                    ),

                "source":
                    "frozen_original",
            })


    return pd.DataFrame(
        rows
    )


# =====================================================================
# 15. SAVE ONE MODEL'S PREDICTIONS
# =====================================================================

def make_prediction_frame(
    samples,
    test_indices,
    truth,
    prediction,
    score,
    cohort,
    model_name,
    fold,
    seed,
    schedule_id,
    gate_value=None,
):

    rows = prediction_rows(
        samples=samples,
        test_indices=test_indices,
        truth=truth,
        prediction=prediction,
        score=score,
        cohort=cohort,
        protocol="phase4_repeated_seed_variability",
        model=model_name,
        fold=fold,
        seed=seed,
        gate_value=gate_value,
    )


    frame = pd.DataFrame(
        rows
    )


    frame[
        "seed_schedule"
    ] = schedule_id

    frame[
        "seed_schedule_index"
    ] = int(
        schedule_id[
            1:
        ]
    )

    frame[
        "seed_offset"
    ] = SEED_SCHEDULES[
        schedule_id
    ]

    frame[
        "seed_source"
    ] = "new_retraining"


    return frame


# =====================================================================
# 16. TRAIN ONE COHORT — RESUMABLE AT FOLD/SCHEDULE LEVEL
# =====================================================================

def train_cohort(
    cohort,
):

    print("\n" + "=" * 105)
    print(
        f"PHASE 4 — {cohort.upper()} REPEATED SEEDS"
    )
    print("=" * 105)


    config = load_config(
        cohort
    )


    (
        samples,
        history_columns,
        assignments,
        y,
        X_history_raw,
        id_to_index,
    ) = load_frozen_cohort(
        cohort
    )


    # -------------------------------------------------------------
    # Determine whether any NEW fold/schedule checkpoint is missing.
    # -------------------------------------------------------------

    missing_jobs = []


    for fold in range(
        1,
        6,
    ):

        for schedule_id in NEW_SCHEDULES:

            checkpoint = (
                CHECKPOINT_DIR
                / (
                    f"{cohort}_"
                    f"{schedule_id}_"
                    f"fold{fold}_predictions.csv"
                )
            )


            epochs_checkpoint = (
                CHECKPOINT_DIR
                / (
                    f"{cohort}_"
                    f"{schedule_id}_"
                    f"fold{fold}_epochs.csv"
                )
            )


            if (
                not checkpoint.exists()
                or
                not epochs_checkpoint.exists()
            ):

                missing_jobs.append(
                    (
                        fold,
                        schedule_id,
                    )
                )


    if not missing_jobs:

        print(
            f"\n✓ All new {cohort} seed/fold checkpoints already exist."
        )

        print(
            "Skipping graph reconstruction and training."
        )

        return


    print(
        f"\nNew training jobs remaining: {len(missing_jobs)}"
    )


    graphs = reconstruct_frozen_graphs(
        cohort,
        config,
        samples,
        assignments,
    )


    # -------------------------------------------------------------
    # LOOP FOLD FIRST.
    #
    # This lets preprocessing be performed once and reused across
    # all four new seed schedules for that fold.
    # -------------------------------------------------------------

    for fold in range(
        1,
        6,
    ):

        schedules_needed = []


        for schedule_id in NEW_SCHEDULES:

            prediction_checkpoint = (
                CHECKPOINT_DIR
                / (
                    f"{cohort}_"
                    f"{schedule_id}_"
                    f"fold{fold}_predictions.csv"
                )
            )


            epochs_checkpoint = (
                CHECKPOINT_DIR
                / (
                    f"{cohort}_"
                    f"{schedule_id}_"
                    f"fold{fold}_epochs.csv"
                )
            )


            if (
                prediction_checkpoint.exists()
                and
                epochs_checkpoint.exists()
            ):

                print(
                    f"✓ {cohort} {schedule_id} fold {fold} "
                    "already completed — skipping."
                )

            else:

                schedules_needed.append(
                    schedule_id
                )


        if not schedules_needed:

            continue


        outer_train = indices_for(
            assignments,
            id_to_index,
            fold,
            "outer",
            "train",
        )


        outer_test = indices_for(
            assignments,
            id_to_index,
            fold,
            "outer",
            "test",
        )


        inner_train = indices_for(
            assignments,
            id_to_index,
            fold,
            "inner",
            "train",
        )


        inner_validation = indices_for(
            assignments,
            id_to_index,
            fold,
            "inner",
            "validation",
        )


        print("\n" + "-" * 105)

        print(
            f"{cohort.upper()} FOLD {fold}"
        )

        print(
            "Outer train:",
            len(
                outer_train
            )
        )

        print(
            "Outer test:",
            len(
                outer_test
            )
        )

        print(
            "Inner train:",
            len(
                inner_train
            )
        )

        print(
            "Inner validation:",
            len(
                inner_validation
            )
        )

        print(
            "New schedules:",
            schedules_needed
        )


        # ---------------------------------------------------------
        # PREPROCESS INNER HISTORY
        # ---------------------------------------------------------

        inner_history_scaler = (
            StandardScaler()
            .fit(
                X_history_raw[
                    inner_train
                ]
            )
        )


        X_inner_train = (
            inner_history_scaler
            .transform(
                X_history_raw[
                    inner_train
                ]
            )
        )


        X_inner_validation = (
            inner_history_scaler
            .transform(
                X_history_raw[
                    inner_validation
                ]
            )
        )


        # ---------------------------------------------------------
        # PREPROCESS INNER GRAPHS
        # ---------------------------------------------------------

        inner_graph_train_raw = full_graph_views(
            graphs,
            inner_train,
        )


        inner_graph_validation_raw = full_graph_views(
            graphs,
            inner_validation,
        )


        (
            inner_graph_train,
            inner_graph_validation,
        ) = preprocess_graph_pair(
            inner_graph_train_raw,
            inner_graph_validation_raw,
            X_inner_train,
            X_inner_validation,
        )


        # ---------------------------------------------------------
        # PREPROCESS OUTER HISTORY
        # ---------------------------------------------------------

        outer_history_scaler = (
            StandardScaler()
            .fit(
                X_history_raw[
                    outer_train
                ]
            )
        )


        X_outer_train = (
            outer_history_scaler
            .transform(
                X_history_raw[
                    outer_train
                ]
            )
        )


        X_outer_test = (
            outer_history_scaler
            .transform(
                X_history_raw[
                    outer_test
                ]
            )
        )


        # ---------------------------------------------------------
        # PREPROCESS OUTER GRAPHS
        # ---------------------------------------------------------

        outer_graph_train_raw = full_graph_views(
            graphs,
            outer_train,
        )


        outer_graph_test_raw = full_graph_views(
            graphs,
            outer_test,
        )


        (
            outer_graph_train,
            outer_graph_test,
        ) = preprocess_graph_pair(
            outer_graph_train_raw,
            outer_graph_test_raw,
            X_outer_train,
            X_outer_test,
        )


        # ---------------------------------------------------------
        # TRAIN EACH NEW SEED SCHEDULE
        # ---------------------------------------------------------

        for schedule_id in schedules_needed:

            seed = seed_for(
                schedule_id,
                fold,
            )


            print("\n" + "." * 95)

            print(
                f"{cohort.upper()} | "
                f"{schedule_id} | "
                f"Fold {fold} | "
                f"Seed {seed}"
            )

            print("." * 95)


            history_factory = (
                lambda:
                HistoryMLP(
                    X_history_raw.shape[
                        1
                    ],
                    int(
                        config[
                            "model"
                        ][
                            "history_hidden_dim"
                        ]
                    ),
                    float(
                        config[
                            "model"
                        ][
                            "history_dropout"
                        ]
                    ),
                )
            )


            positional_factory = (
                lambda:
                HybridGNN(
                    4,
                    3,
                    X_history_raw.shape[
                        1
                    ],
                    config,
                    pooling="positional_add",
                )
            )


            globalmean_factory = (
                lambda:
                HybridGNN(
                    4,
                    3,
                    X_history_raw.shape[
                        1
                    ],
                    config,
                    pooling="global_mean",
                )
            )


            fold_prediction_frames = []
            epoch_rows = []
            runtime_rows = []


            # =====================================================
            # A-MLP — RESELECT EPOCH FOR THIS SEED
            # =====================================================

            start = time.perf_counter()


            a_selection = select_history_epoch(
                history_factory,
                X_inner_train,
                y[
                    inner_train
                ],
                X_inner_validation,
                y[
                    inner_validation
                ],
                config,
                seed,
                device,
            )


            selection_seconds = (
                time.perf_counter()
                - start
            )


            start = time.perf_counter()


            a_model = fit_history_model(
                history_factory,
                X_outer_train,
                y[
                    outer_train
                ],
                a_selection.best_epoch,
                config,
                seed,
                device,
            )


            fit_seconds = (
                time.perf_counter()
                - start
            )


            start = time.perf_counter()


            (
                a_pred,
                a_score,
            ) = predict_history_model(
                a_model,
                X_outer_test,
                device,
            )


            predict_seconds = (
                time.perf_counter()
                - start
            )


            a_frame = make_prediction_frame(
                samples,
                outer_test,
                y[
                    outer_test
                ],
                a_pred,
                a_score,
                cohort,
                "A-MLP",
                fold,
                seed,
                schedule_id,
                gate_value=None,
            )


            fold_prediction_frames.append(
                a_frame
            )


            epoch_rows.append({
                "cohort":
                    cohort,

                "seed_schedule":
                    schedule_id,

                "seed_schedule_index":
                    int(
                        schedule_id[
                            1:
                        ]
                    ),

                "fold":
                    fold,

                "seed":
                    seed,

                "model":
                    "A-MLP",

                "best_epoch":
                    int(
                        a_selection.best_epoch
                    ),

                "validation_macro_f1":
                    float(
                        a_selection.best_metric
                    ),

                "source":
                    "new_retraining",
            })


            runtime_rows.append({
                "cohort":
                    cohort,

                "seed_schedule":
                    schedule_id,

                "fold":
                    fold,

                "seed":
                    seed,

                "model":
                    "A-MLP",

                "selection_seconds":
                    selection_seconds,

                "fit_seconds":
                    fit_seconds,

                "predict_seconds":
                    predict_seconds,

                "total_seconds":
                    (
                        selection_seconds
                        + fit_seconds
                        + predict_seconds
                    ),
            })


            del a_model
            gc.collect()
            torch.cuda.empty_cache()


            # =====================================================
            # C-HYBRID POSITIONAL — RESELECT EPOCH FOR THIS SEED
            # =====================================================

            start = time.perf_counter()


            p_selection = select_graph_epoch(
                positional_factory,
                inner_graph_train,
                inner_graph_validation,
                y[
                    inner_validation
                ],
                config,
                seed,
                device,
            )


            selection_seconds = (
                time.perf_counter()
                - start
            )


            start = time.perf_counter()


            p_model = fit_graph_model(
                positional_factory,
                outer_graph_train,
                p_selection.best_epoch,
                config,
                seed,
                device,
            )


            fit_seconds = (
                time.perf_counter()
                - start
            )


            start = time.perf_counter()


            (
                p_pred,
                p_score,
            ) = predict_graph_model(
                p_model,
                outer_graph_test,
                config,
                device,
            )


            predict_seconds = (
                time.perf_counter()
                - start
            )


            p_gate = (
                p_model
                .gate_value()
            )


            p_frame = make_prediction_frame(
                samples,
                outer_test,
                y[
                    outer_test
                ],
                p_pred,
                p_score,
                cohort,
                "C-Hybrid-positional",
                fold,
                seed,
                schedule_id,
                gate_value=p_gate,
            )


            fold_prediction_frames.append(
                p_frame
            )


            epoch_rows.append({
                "cohort":
                    cohort,

                "seed_schedule":
                    schedule_id,

                "seed_schedule_index":
                    int(
                        schedule_id[
                            1:
                        ]
                    ),

                "fold":
                    fold,

                "seed":
                    seed,

                "model":
                    "C-Hybrid-positional",

                "best_epoch":
                    int(
                        p_selection.best_epoch
                    ),

                "validation_macro_f1":
                    float(
                        p_selection.best_metric
                    ),

                "source":
                    "new_retraining",
            })


            runtime_rows.append({
                "cohort":
                    cohort,

                "seed_schedule":
                    schedule_id,

                "fold":
                    fold,

                "seed":
                    seed,

                "model":
                    "C-Hybrid-positional",

                "selection_seconds":
                    selection_seconds,

                "fit_seconds":
                    fit_seconds,

                "predict_seconds":
                    predict_seconds,

                "total_seconds":
                    (
                        selection_seconds
                        + fit_seconds
                        + predict_seconds
                    ),
            })


            print(
                "  A-MLP epoch:",
                a_selection.best_epoch
            )

            print(
                "  C-Hybrid positional epoch:",
                p_selection.best_epoch
            )

            print(
                "  Positional lambda:",
                round(
                    p_gate,
                    6,
                )
            )


            del p_model
            gc.collect()
            torch.cuda.empty_cache()


            # =====================================================
            # ELITE ONLY: C-HYBRID GLOBAL-MEAN POOLING
            # =====================================================

            if cohort == "elite":

                start = time.perf_counter()


                g_selection = select_graph_epoch(
                    globalmean_factory,
                    inner_graph_train,
                    inner_graph_validation,
                    y[
                        inner_validation
                    ],
                    config,
                    seed,
                    device,
                )


                selection_seconds = (
                    time.perf_counter()
                    - start
                )


                start = time.perf_counter()


                g_model = fit_graph_model(
                    globalmean_factory,
                    outer_graph_train,
                    g_selection.best_epoch,
                    config,
                    seed,
                    device,
                )


                fit_seconds = (
                    time.perf_counter()
                    - start
                )


                start = time.perf_counter()


                (
                    g_pred,
                    g_score,
                ) = predict_graph_model(
                    g_model,
                    outer_graph_test,
                    config,
                    device,
                )


                predict_seconds = (
                    time.perf_counter()
                    - start
                )


                g_gate = (
                    g_model
                    .gate_value()
                )


                g_frame = make_prediction_frame(
                    samples,
                    outer_test,
                    y[
                        outer_test
                    ],
                    g_pred,
                    g_score,
                    cohort,
                    "C-Hybrid-globalmean",
                    fold,
                    seed,
                    schedule_id,
                    gate_value=g_gate,
                )


                fold_prediction_frames.append(
                    g_frame
                )


                epoch_rows.append({
                    "cohort":
                        cohort,

                    "seed_schedule":
                        schedule_id,

                    "seed_schedule_index":
                        int(
                            schedule_id[
                                1:
                            ]
                        ),

                    "fold":
                        fold,

                    "seed":
                        seed,

                    "model":
                        "C-Hybrid-globalmean",

                    "best_epoch":
                        int(
                            g_selection.best_epoch
                        ),

                    "validation_macro_f1":
                        float(
                            g_selection.best_metric
                        ),

                    "source":
                        "new_retraining",
                })


                runtime_rows.append({
                    "cohort":
                        cohort,

                    "seed_schedule":
                        schedule_id,

                    "fold":
                        fold,

                    "seed":
                        seed,

                    "model":
                        "C-Hybrid-globalmean",

                    "selection_seconds":
                        selection_seconds,

                    "fit_seconds":
                        fit_seconds,

                    "predict_seconds":
                        predict_seconds,

                    "total_seconds":
                        (
                            selection_seconds
                            + fit_seconds
                            + predict_seconds
                        ),
                })


                print(
                    "  C-Hybrid globalmean epoch:",
                    g_selection.best_epoch
                )

                print(
                    "  Globalmean lambda:",
                    round(
                        g_gate,
                        6,
                    )
                )


                del g_model
                gc.collect()
                torch.cuda.empty_cache()


            # =====================================================
            # VALIDATE + WRITE CHECKPOINT AT END OF FOLD/SCHEDULE
            # =====================================================

            fold_predictions = pd.concat(
                fold_prediction_frames,
                ignore_index=True,
                sort=False,
            )


            expected_models = EXPECTED[
                cohort
            ][
                "models"
            ]


            actual_models = sorted(
                fold_predictions[
                    "model"
                ].unique()
                .tolist()
            )


            if sorted(
                expected_models
            ) != actual_models:

                raise RuntimeError(
                    f"{cohort} {schedule_id} fold {fold}: "
                    "model set mismatch."
                )


            expected_test_n = len(
                outer_test
            )


            for model_name in expected_models:

                model_block = fold_predictions[
                    fold_predictions[
                        "model"
                    ].eq(
                        model_name
                    )
                ]


                if len(
                    model_block
                ) != expected_test_n:

                    raise RuntimeError(
                        f"{cohort} {schedule_id} fold {fold} "
                        f"{model_name}: incorrect prediction count."
                    )


            prediction_checkpoint = (
                CHECKPOINT_DIR
                / (
                    f"{cohort}_"
                    f"{schedule_id}_"
                    f"fold{fold}_predictions.csv"
                )
            )


            epochs_checkpoint = (
                CHECKPOINT_DIR
                / (
                    f"{cohort}_"
                    f"{schedule_id}_"
                    f"fold{fold}_epochs.csv"
                )
            )


            runtime_checkpoint = (
                CHECKPOINT_DIR
                / (
                    f"{cohort}_"
                    f"{schedule_id}_"
                    f"fold{fold}_runtime.csv"
                )
            )


            fold_predictions.to_csv(
                prediction_checkpoint,
                index=False,
            )


            pd.DataFrame(
                epoch_rows
            ).to_csv(
                epochs_checkpoint,
                index=False,
            )


            pd.DataFrame(
                runtime_rows
            ).to_csv(
                runtime_checkpoint,
                index=False,
            )


            print(
                f"✓ CHECKPOINT SAVED: "
                f"{cohort} {schedule_id} fold {fold}"
            )


        # Free scaled fold copies before next fold.
        del inner_graph_train_raw
        del inner_graph_validation_raw
        del inner_graph_train
        del inner_graph_validation

        del outer_graph_train_raw
        del outer_graph_test_raw
        del outer_graph_train
        del outer_graph_test

        gc.collect()
        torch.cuda.empty_cache()


    del graphs
    gc.collect()
    torch.cuda.empty_cache()


# =====================================================================
# 17. CREATE S0 FROZEN CHECKPOINTS
# =====================================================================

print("\n" + "=" * 105)
print("PREPARING FROZEN ORIGINAL S0")
print("=" * 105)


elite_S0 = build_original_S0_predictions(
    "elite"
)


global_S0 = build_original_S0_predictions(
    "global"
)


elite_S0_epochs = original_S0_epoch_rows(
    "elite"
)


global_S0_epochs = original_S0_epoch_rows(
    "global"
)


print(
    "✓ Original S0 Elite predictions loaded."
)

print(
    "✓ Original S0 Global predictions loaded."
)


# =====================================================================
# 18. RUN NEW ELITE SEEDS
# =====================================================================

train_cohort(
    "elite"
)


# =====================================================================
# 19. RUN NEW GLOBAL SEEDS
# =====================================================================

train_cohort(
    "global"
)


# =====================================================================
# 20. COLLECT ALL PREDICTIONS
# =====================================================================

prediction_frames = [
    elite_S0,
    global_S0,
]


epoch_frames = [
    elite_S0_epochs,
    global_S0_epochs,
]


runtime_frames = []


for cohort in [
    "elite",
    "global",
]:

    for schedule_id in NEW_SCHEDULES:

        for fold in range(
            1,
            6,
        ):

            prediction_path = (
                CHECKPOINT_DIR
                / (
                    f"{cohort}_"
                    f"{schedule_id}_"
                    f"fold{fold}_predictions.csv"
                )
            )


            epoch_path = (
                CHECKPOINT_DIR
                / (
                    f"{cohort}_"
                    f"{schedule_id}_"
                    f"fold{fold}_epochs.csv"
                )
            )


            runtime_path = (
                CHECKPOINT_DIR
                / (
                    f"{cohort}_"
                    f"{schedule_id}_"
                    f"fold{fold}_runtime.csv"
                )
            )


            if (
                not prediction_path.exists()
                or
                not epoch_path.exists()
            ):

                raise RuntimeError(
                    "Phase 4 is incomplete.\n"
                    f"Missing checkpoint for "
                    f"{cohort} {schedule_id} fold {fold}."
                )


            prediction_frames.append(
                pd.read_csv(
                    prediction_path
                )
            )


            epoch_frames.append(
                pd.read_csv(
                    epoch_path
                )
            )


            if runtime_path.exists():

                runtime_frames.append(
                    pd.read_csv(
                        runtime_path
                    )
                )


all_predictions = pd.concat(
    prediction_frames,
    ignore_index=True,
    sort=False,
)


all_epochs = pd.concat(
    epoch_frames,
    ignore_index=True,
    sort=False,
)


all_runtime = (
    pd.concat(
        runtime_frames,
        ignore_index=True,
        sort=False,
    )

    if runtime_frames

    else pd.DataFrame()
)


# =====================================================================
# 21. FINAL PREDICTION-INTEGRITY AUDIT
# =====================================================================

for cohort in [
    "elite",
    "global",
]:

    expected_n = EXPECTED[
        cohort
    ][
        "oof_n"
    ]


    expected_models = EXPECTED[
        cohort
    ][
        "models"
    ]


    cohort_predictions = all_predictions[
        all_predictions[
            "cohort"
        ].eq(
            cohort
        )
    ]


    reference_ids = None


    for schedule_id in SEED_SCHEDULES:

        schedule_block = cohort_predictions[
            cohort_predictions[
                "seed_schedule"
            ].eq(
                schedule_id
            )
        ]


        for model_name in expected_models:

            model_block = schedule_block[
                schedule_block[
                    "model"
                ].eq(
                    model_name
                )
            ]


            if len(
                model_block
            ) != expected_n:

                raise RuntimeError(
                    f"{cohort} {schedule_id} {model_name}: "
                    f"expected {expected_n}, "
                    f"got {len(model_block)}."
                )


            if model_block.duplicated(
                [
                    "match_id",
                ]
            ).any():

                raise RuntimeError(
                    f"Duplicate predictions in "
                    f"{cohort} {schedule_id} {model_name}."
                )


            ids = set(
                model_block[
                    "match_id"
                ].astype(
                    int
                )
            )


            if reference_ids is None:

                reference_ids = ids


            elif ids != reference_ids:

                raise RuntimeError(
                    f"OOF test IDs differ across seeds/models "
                    f"for {cohort}."
                )


    print(
        f"✓ {cohort.upper()} prediction integrity passed: "
        f"5 schedules x {len(expected_models)} models x "
        f"{expected_n} OOF observations."
    )


# =====================================================================
# 22. METRIC FUNCTION
# =====================================================================

def metric_row(
    truth,
    pred,
    score,
):

    truth = np.asarray(
        truth,
        dtype=int,
    )

    pred = np.asarray(
        pred,
        dtype=int,
    )

    score = np.asarray(
        score,
        dtype=float,
    )


    return {
        "n":
            int(
                len(
                    truth
                )
            ),

        "accuracy":
            float(
                accuracy_score(
                    truth,
                    pred,
                )
            ),

        "balanced_accuracy":
            float(
                balanced_accuracy_score(
                    truth,
                    pred,
                )
            ),

        "macro_f1":
            float(
                f1_score(
                    truth,
                    pred,
                    average="macro",
                    labels=[
                        0,
                        1,
                    ],
                    zero_division=0,
                )
            ),

        "win_precision":
            float(
                precision_score(
                    truth,
                    pred,
                    pos_label=1,
                    zero_division=0,
                )
            ),

        "win_recall":
            float(
                recall_score(
                    truth,
                    pred,
                    pos_label=1,
                    zero_division=0,
                )
            ),

        "roc_auc":
            (
                float(
                    roc_auc_score(
                        truth,
                        score,
                    )
                )

                if len(
                    np.unique(
                        truth
                    )
                ) == 2

                else np.nan
            ),
    }


# =====================================================================
# 23. POOLED METRICS FOR EACH COMPLETE SEED SCHEDULE
# =====================================================================

schedule_metric_rows = []


for (
    cohort,
    schedule_id,
    model_name,
), group in all_predictions.groupby(
    [
        "cohort",
        "seed_schedule",
        "model",
    ]
):

    row = {
        "cohort":
            cohort,

        "seed_schedule":
            schedule_id,

        "seed_schedule_index":
            int(
                schedule_id[
                    1:
                ]
            ),

        "model":
            model_name,
    }


    row.update(
        metric_row(
            group[
                "true_label"
            ].to_numpy(),

            group[
                "predicted_class"
            ].to_numpy(),

            group[
                "predicted_score"
            ].to_numpy(),
        )
    )


    schedule_metric_rows.append(
        row
    )


schedule_metrics = pd.DataFrame(
    schedule_metric_rows
).sort_values(
    [
        "cohort",
        "seed_schedule_index",
        "model",
    ]
)


# =====================================================================
# 24. EFFECT SIZE FOR EACH SEED SCHEDULE
# =====================================================================

effect_rows = []


for schedule_id in SEED_SCHEDULES:

    # -------------------------------------------------------------
    # ELITE: MAIN HYBRID COMPARISON
    # -------------------------------------------------------------

    elite_metrics = schedule_metrics[
        schedule_metrics[
            "cohort"
        ].eq(
            "elite"
        )
        &
        schedule_metrics[
            "seed_schedule"
        ].eq(
            schedule_id
        )
    ].set_index(
        "model"
    )


    pos_f1 = float(
        elite_metrics.loc[
            "C-Hybrid-positional",
            "macro_f1",
        ]
    )

    mlp_f1 = float(
        elite_metrics.loc[
            "A-MLP",
            "macro_f1",
        ]
    )

    globalmean_f1 = float(
        elite_metrics.loc[
            "C-Hybrid-globalmean",
            "macro_f1",
        ]
    )


    pos_acc = float(
        elite_metrics.loc[
            "C-Hybrid-positional",
            "accuracy",
        ]
    )

    mlp_acc = float(
        elite_metrics.loc[
            "A-MLP",
            "accuracy",
        ]
    )

    globalmean_acc = float(
        elite_metrics.loc[
            "C-Hybrid-globalmean",
            "accuracy",
        ]
    )


    effect_rows.append({
        "cohort":
            "elite",

        "seed_schedule":
            schedule_id,

        "seed_schedule_index":
            int(
                schedule_id[
                    1:
                ]
            ),

        "comparison":
            "C-Hybrid-positional vs A-MLP",

        "comparison_role":
            "main_hybrid_comparison",

        "model_a":
            "C-Hybrid-positional",

        "model_b":
            "A-MLP",

        "model_a_macro_f1":
            pos_f1,

        "model_b_macro_f1":
            mlp_f1,

        "macro_f1_difference_pp":
            100.0
            * (
                pos_f1
                - mlp_f1
            ),

        "accuracy_difference_pp":
            100.0
            * (
                pos_acc
                - mlp_acc
            ),
    })


    # -------------------------------------------------------------
    # ELITE: POOLING ABLATION
    # -------------------------------------------------------------

    effect_rows.append({
        "cohort":
            "elite",

        "seed_schedule":
            schedule_id,

        "seed_schedule_index":
            int(
                schedule_id[
                    1:
                ]
            ),

        "comparison":
            (
                "C-Hybrid-positional "
                "vs C-Hybrid-globalmean"
            ),

        "comparison_role":
            "pooling_ablation",

        "model_a":
            "C-Hybrid-positional",

        "model_b":
            "C-Hybrid-globalmean",

        "model_a_macro_f1":
            pos_f1,

        "model_b_macro_f1":
            globalmean_f1,

        "macro_f1_difference_pp":
            100.0
            * (
                pos_f1
                - globalmean_f1
            ),

        "accuracy_difference_pp":
            100.0
            * (
                pos_acc
                - globalmean_acc
            ),
    })


    # -------------------------------------------------------------
    # GLOBAL: MAIN HYBRID COMPARISON
    # -------------------------------------------------------------

    global_metrics = schedule_metrics[
        schedule_metrics[
            "cohort"
        ].eq(
            "global"
        )
        &
        schedule_metrics[
            "seed_schedule"
        ].eq(
            schedule_id
        )
    ].set_index(
        "model"
    )


    global_pos_f1 = float(
        global_metrics.loc[
            "C-Hybrid-positional",
            "macro_f1",
        ]
    )

    global_mlp_f1 = float(
        global_metrics.loc[
            "A-MLP",
            "macro_f1",
        ]
    )


    global_pos_acc = float(
        global_metrics.loc[
            "C-Hybrid-positional",
            "accuracy",
        ]
    )

    global_mlp_acc = float(
        global_metrics.loc[
            "A-MLP",
            "accuracy",
        ]
    )


    effect_rows.append({
        "cohort":
            "global",

        "seed_schedule":
            schedule_id,

        "seed_schedule_index":
            int(
                schedule_id[
                    1:
                ]
            ),

        "comparison":
            "C-Hybrid-positional vs A-MLP",

        "comparison_role":
            "main_hybrid_comparison",

        "model_a":
            "C-Hybrid-positional",

        "model_b":
            "A-MLP",

        "model_a_macro_f1":
            global_pos_f1,

        "model_b_macro_f1":
            global_mlp_f1,

        "macro_f1_difference_pp":
            100.0
            * (
                global_pos_f1
                - global_mlp_f1
            ),

        "accuracy_difference_pp":
            100.0
            * (
                global_pos_acc
                - global_mlp_acc
            ),
    })


effect_by_seed = pd.DataFrame(
    effect_rows
).sort_values(
    [
        "cohort",
        "comparison_role",
        "seed_schedule_index",
    ]
)


# =====================================================================
# 25. MODEL-LEVEL TRAINING VARIABILITY SUMMARY
# =====================================================================

model_variability_rows = []


for (
    cohort,
    model_name,
), group in schedule_metrics.groupby(
    [
        "cohort",
        "model",
    ]
):

    macro_values = group[
        "macro_f1"
    ].to_numpy(
        dtype=float
    )


    accuracy_values = group[
        "accuracy"
    ].to_numpy(
        dtype=float
    )


    model_variability_rows.append({
        "cohort":
            cohort,

        "model":
            model_name,

        "seed_schedule_count":
            int(
                len(
                    group
                )
            ),

        "macro_f1_mean":
            float(
                macro_values.mean()
            ),

        "macro_f1_sd":
            float(
                macro_values.std(
                    ddof=1
                )
            ),

        "macro_f1_min":
            float(
                macro_values.min()
            ),

        "macro_f1_max":
            float(
                macro_values.max()
            ),

        "macro_f1_range_pp":
            100.0
            * float(
                macro_values.max()
                - macro_values.min()
            ),

        "accuracy_mean":
            float(
                accuracy_values.mean()
            ),

        "accuracy_sd":
            float(
                accuracy_values.std(
                    ddof=1
                )
            ),

        "accuracy_min":
            float(
                accuracy_values.min()
            ),

        "accuracy_max":
            float(
                accuracy_values.max()
            ),
    })


model_variability_summary = pd.DataFrame(
    model_variability_rows
)


# =====================================================================
# 26. EFFECT-LEVEL TRAINING VARIABILITY SUMMARY
# =====================================================================

effect_variability_rows = []


for (
    cohort,
    comparison,
    comparison_role,
), group in effect_by_seed.groupby(
    [
        "cohort",
        "comparison",
        "comparison_role",
    ]
):

    f1_deltas = group[
        "macro_f1_difference_pp"
    ].to_numpy(
        dtype=float
    )


    acc_deltas = group[
        "accuracy_difference_pp"
    ].to_numpy(
        dtype=float
    )


    effect_variability_rows.append({
        "cohort":
            cohort,

        "comparison":
            comparison,

        "comparison_role":
            comparison_role,

        "seed_schedule_count":
            int(
                len(
                    f1_deltas
                )
            ),

        "macro_f1_difference_mean_pp":
            float(
                f1_deltas.mean()
            ),

        "macro_f1_difference_sd_pp":
            float(
                f1_deltas.std(
                    ddof=1
                )
            ),

        "macro_f1_difference_median_pp":
            float(
                np.median(
                    f1_deltas
                )
            ),

        "macro_f1_difference_min_pp":
            float(
                f1_deltas.min()
            ),

        "macro_f1_difference_max_pp":
            float(
                f1_deltas.max()
            ),

        "macro_f1_difference_range_pp":
            float(
                f1_deltas.max()
                - f1_deltas.min()
            ),

        "positive_seed_schedules":
            int(
                np.sum(
                    f1_deltas
                    > 0
                )
            ),

        "negative_seed_schedules":
            int(
                np.sum(
                    f1_deltas
                    < 0
                )
            ),

        "zero_seed_schedules":
            int(
                np.sum(
                    f1_deltas
                    == 0
                )
            ),

        "direction_consistency_fraction":
            float(
                max(
                    np.mean(
                        f1_deltas
                        > 0
                    ),
                    np.mean(
                        f1_deltas
                        < 0
                    ),
                )
            ),

        "accuracy_difference_mean_pp":
            float(
                acc_deltas.mean()
            ),

        "accuracy_difference_sd_pp":
            float(
                acc_deltas.std(
                    ddof=1
                )
            ),

        "accuracy_difference_min_pp":
            float(
                acc_deltas.min()
            ),

        "accuracy_difference_max_pp":
            float(
                acc_deltas.max()
            ),
    })


effect_variability_summary = pd.DataFrame(
    effect_variability_rows
)


# =====================================================================
# 27. SELECTED-EPOCH VARIABILITY
# =====================================================================

epoch_variability_rows = []


for (
    cohort,
    model_name,
), group in all_epochs.groupby(
    [
        "cohort",
        "model",
    ]
):

    epochs = group[
        "best_epoch"
    ].astype(
        float
    ).to_numpy()


    epoch_variability_rows.append({
        "cohort":
            cohort,

        "model":
            model_name,

        "selection_count":
            int(
                len(
                    epochs
                )
            ),

        "best_epoch_mean":
            float(
                epochs.mean()
            ),

        "best_epoch_sd":
            float(
                epochs.std(
                    ddof=1
                )
            ),

        "best_epoch_min":
            int(
                epochs.min()
            ),

        "best_epoch_max":
            int(
                epochs.max()
            ),
    })


epoch_variability_summary = pd.DataFrame(
    epoch_variability_rows
)


# =====================================================================
# 28. GATE VARIABILITY
# =====================================================================

gate_rows = (
    all_predictions[
        all_predictions[
            "model"
        ].str.startswith(
            "C-Hybrid"
        )
    ]
    [
        [
            "cohort",
            "seed_schedule",
            "fold",
            "seed",
            "model",
            "gate_value",
        ]
    ]
    .drop_duplicates()
    .dropna(
        subset=[
            "gate_value"
        ]
    )
)


gate_variability_rows = []


for (
    cohort,
    model_name,
), group in gate_rows.groupby(
    [
        "cohort",
        "model",
    ]
):

    values = group[
        "gate_value"
    ].astype(
        float
    ).to_numpy()


    gate_variability_rows.append({
        "cohort":
            cohort,

        "model":
            model_name,

        "gate_observation_count":
            int(
                len(
                    values
                )
            ),

        "gate_mean":
            float(
                values.mean()
            ),

        "gate_sd":
            float(
                values.std(
                    ddof=1
                )
            ),

        "gate_min":
            float(
                values.min()
            ),

        "gate_max":
            float(
                values.max()
            ),

        "interpretation_note":
            (
                "Fusion coefficient only; "
                "must not be interpreted as percentage modality importance."
            ),
    })


gate_variability_summary = pd.DataFrame(
    gate_variability_rows
)


# =====================================================================
# 29. PARAMETER COUNTS
# =====================================================================

elite_config = load_config(
    "elite"
)


parameter_rows = [
    {
        "model":
            "A-MLP",

        "trainable_parameters":
            int(
                sum(
                    parameter.numel()
                    for parameter in HistoryMLP(
                        16,
                        int(
                            elite_config[
                                "model"
                            ][
                                "history_hidden_dim"
                            ]
                        ),
                        float(
                            elite_config[
                                "model"
                            ][
                                "history_dropout"
                            ]
                        ),
                    ).parameters()
                    if parameter.requires_grad
                )
            ),
    },

    {
        "model":
            "C-Hybrid-positional",

        "trainable_parameters":
            int(
                sum(
                    parameter.numel()
                    for parameter in HybridGNN(
                        4,
                        3,
                        16,
                        elite_config,
                        pooling="positional_add",
                    ).parameters()
                    if parameter.requires_grad
                )
            ),
    },

    {
        "model":
            "C-Hybrid-globalmean",

        "trainable_parameters":
            int(
                sum(
                    parameter.numel()
                    for parameter in HybridGNN(
                        4,
                        3,
                        16,
                        elite_config,
                        pooling="global_mean",
                    ).parameters()
                    if parameter.requires_grad
                )
            ),
    },
]


parameter_counts = pd.DataFrame(
    parameter_rows
)


# =====================================================================
# 30. SAVE FINAL OUTPUTS
# =====================================================================

save_environment(
    RESULT_DIR
    / "environment.json"
)


all_predictions.to_csv(
    RESULT_DIR
    / "all_repeated_seed_oof_predictions.csv",
    index=False,
)


all_epochs.to_csv(
    RESULT_DIR
    / "all_selected_epochs.csv",
    index=False,
)


if len(
    all_runtime
):

    all_runtime.to_csv(
        RESULT_DIR
        / "runtime_summary.csv",
        index=False,
    )


schedule_metrics.to_csv(
    RESULT_DIR
    / "metrics_by_seed_schedule.csv",
    index=False,
)


effect_by_seed.to_csv(
    RESULT_DIR
    / "effects_by_seed_schedule.csv",
    index=False,
)


model_variability_summary.to_csv(
    RESULT_DIR
    / "model_variability_summary.csv",
    index=False,
)


effect_variability_summary.to_csv(
    RESULT_DIR
    / "effect_variability_summary.csv",
    index=False,
)


epoch_variability_summary.to_csv(
    RESULT_DIR
    / "epoch_variability_summary.csv",
    index=False,
)


gate_rows.to_csv(
    RESULT_DIR
    / "gate_values_by_seed_and_fold.csv",
    index=False,
)


gate_variability_summary.to_csv(
    RESULT_DIR
    / "gate_variability_summary.csv",
    index=False,
)


parameter_counts.to_csv(
    RESULT_DIR
    / "parameter_counts.csv",
    index=False,
)


# =====================================================================
# 31. FROZEN PROTOCOL / VALIDATION FILE
# =====================================================================

protocol = {
    "phase":
        "Phase 4 - repeated-training-seed variability",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "repository_commit":
        FROZEN_COMMIT,

    "seed_schedule_count":
        5,

    "seed_schedules":
        {
            schedule_id: [
                seed_for(
                    schedule_id,
                    fold,
                )
                for fold in range(
                    1,
                    6,
                )
            ]

            for schedule_id
            in SEED_SCHEDULES
        },

    "S0":
        (
            "exact frozen original submitted predictions; "
            "not retrained"
        ),

    "S1_to_S4":
        (
            "four additional complete retraining schedules"
        ),

    "elite_models": [
        "A-MLP",
        "C-Hybrid-positional",
        "C-Hybrid-globalmean",
    ],

    "global_models": [
        "A-MLP",
        "C-Hybrid-positional",
    ],

    "elite_oof_n_per_model_per_schedule":
        454,

    "global_oof_n_per_model_per_schedule":
        1745,

    "early_stopping":
        (
            "reselected independently for every new seed, "
            "fold and model using frozen inner chronological validation"
        ),

    "preprocessing":
        (
            "refit only on corresponding frozen training partition "
            "for each fold; unchanged across seed schedules"
        ),

    "primary_training_variability_questions": [
        "Elite C-Hybrid-positional vs A-MLP",
        "Global C-Hybrid-positional vs A-MLP",
        (
            "Elite C-Hybrid-positional "
            "vs C-Hybrid-globalmean"
        ),
    ],

    "summary_statistics":
        [
            "mean",
            "sample standard deviation",
            "median",
            "minimum",
            "maximum",
            "range",
            "effect direction count",
        ],

    "inference_note":
        (
            "Five-seed analysis is descriptive training-stability "
            "analysis; it is not treated as a new hypothesis-test family."
        ),

    "claim_rule":
        (
            "Report all five schedules. Do not select, discard, "
            "or rerun seeds because of performance direction."
        ),
}


(
    RESULT_DIR
    / "phase4_protocol.json"
).write_text(
    json.dumps(
        protocol,
        indent=2,
    ),
    encoding="utf-8",
)


validation = {
    "repository_commit_verified":
        True,

    "elite_seed_schedules":
        5,

    "global_seed_schedules":
        5,

    "elite_oof_n":
        454,

    "global_oof_n":
        1745,

    "same_folds_across_seeds":
        True,

    "same_features_across_seeds":
        True,

    "same_architecture_across_seeds":
        True,

    "same_optimizer_across_seeds":
        True,

    "same_test_observations_across_seeds":
        True,

    "early_stopping_reselected_for_new_seeds":
        True,

    "new_seed_schedules_trained":
        4,

    "frozen_original_schedule_included":
        True,
}


(
    RESULT_DIR
    / "phase4_validation.json"
).write_text(
    json.dumps(
        validation,
        indent=2,
    ),
    encoding="utf-8",
)


# =====================================================================
# 32. HUMAN-READABLE README
# =====================================================================

readme_text = """
IEEE ACCESS R1 — PHASE 4
REPEATED-TRAINING-SEED VARIABILITY
==================================

Purpose
-------
Quantify neural-training stochasticity that is not captured by
resampling fixed test predictions.

Seed schedules
--------------
Five complete seed schedules are reported.

S0:
142, 242, 342, 442, 542
This is the exact frozen original experiment.

S1:
1142, 1242, 1342, 1442, 1542

S2:
2142, 2242, 2342, 2442, 2542

S3:
3142, 3242, 3342, 3442, 3542

S4:
4142, 4242, 4342, 4442, 4542

S1-S4 are independent additional training repetitions.

For every new seed/fold/model:
1. model parameters are reinitialized;
2. training batches are reshuffled according to the seed;
3. dropout stochasticity follows the new seed;
4. the inner-validation early-stopping epoch is reselected;
5. the model is retrained on the full outer training partition;
6. predictions are generated on the exact same frozen OOF test fold.

Elite analysis
--------------
A-MLP
C-Hybrid-positional
C-Hybrid-globalmean

Questions:
- Is C-Hybrid vs A-MLP stable to training seed?
- Is the positional-vs-global-mean pooling effect stable to seed?

Global analysis
---------------
A-MLP
C-Hybrid-positional

Question:
- Is the main Global Hybrid-vs-MLP advantage stable to seed?

Reporting
---------
For each model:
- macro-F1 mean
- sample SD
- min/max
- range

For each comparison:
- difference for every seed schedule
- mean difference
- sample SD
- min/max difference
- number of schedules favoring each model

Important interpretation
------------------------
This is a limited repeated-seed stability analysis, not a search for the
best-performing seed and not a replacement for observation-level or
dependency-aware uncertainty analyses.

All five schedules must be reported, regardless of direction.
""".strip()


(
    RESULT_DIR
    / "README_PHASE4.txt"
).write_text(
    readme_text,
    encoding="utf-8",
)


# =====================================================================
# 33. PRINT MAIN RESULTS
# =====================================================================

print("\n" + "=" * 110)
print("PHASE 4 — METRICS BY SEED SCHEDULE")
print("=" * 110)


with pd.option_context(
    "display.max_columns",
    None,
    "display.width",
    250,
):

    print(
        schedule_metrics[
            [
                "cohort",
                "seed_schedule",
                "model",
                "n",
                "accuracy",
                "macro_f1",
                "roc_auc",
            ]
        ].to_string(
            index=False
        )
    )


print("\n" + "=" * 110)
print("PHASE 4 — EFFECTS BY SEED SCHEDULE")
print("=" * 110)


print(
    effect_by_seed[
        [
            "cohort",
            "seed_schedule",
            "comparison",
            "macro_f1_difference_pp",
            "accuracy_difference_pp",
        ]
    ].to_string(
        index=False
    )
)


print("\n" + "=" * 110)
print("PHASE 4 — MODEL VARIABILITY")
print("=" * 110)


print(
    model_variability_summary[
        [
            "cohort",
            "model",
            "seed_schedule_count",
            "macro_f1_mean",
            "macro_f1_sd",
            "macro_f1_min",
            "macro_f1_max",
            "macro_f1_range_pp",
        ]
    ].to_string(
        index=False
    )
)


print("\n" + "=" * 110)
print("PHASE 4 — COMPARISON / EFFECT VARIABILITY")
print("=" * 110)


print(
    effect_variability_summary[
        [
            "cohort",
            "comparison",
            "seed_schedule_count",
            "macro_f1_difference_mean_pp",
            "macro_f1_difference_sd_pp",
            "macro_f1_difference_min_pp",
            "macro_f1_difference_max_pp",
            "positive_seed_schedules",
            "negative_seed_schedules",
            "direction_consistency_fraction",
        ]
    ].to_string(
        index=False
    )
)


print("\n" + "=" * 110)
print("SELECTED-EPOCH VARIABILITY")
print("=" * 110)


print(
    epoch_variability_summary
    .to_string(
        index=False
    )
)


# =====================================================================
# 34. AUTOMATIC INTERPRETATION FLAGS
# =====================================================================

print("\n" + "=" * 110)
print("AUTOMATIC STABILITY INTERPRETATION")
print("=" * 110)


for _, row in effect_variability_summary.iterrows():

    print(
        f"\n{row['cohort'].upper()} — "
        f"{row['comparison']}"
    )

    print(
        "  Mean macro-F1 difference:",
        f"{row['macro_f1_difference_mean_pp']:+.3f} pp"
    )

    print(
        "  SD across seed schedules:",
        f"{row['macro_f1_difference_sd_pp']:.3f} pp"
    )

    print(
        "  Range:",
        (
            f"[{row['macro_f1_difference_min_pp']:+.3f}, "
            f"{row['macro_f1_difference_max_pp']:+.3f}] pp"
        ),
    )

    print(
        "  Positive schedules:",
        int(
            row[
                "positive_seed_schedules"
            ]
        ),
        "/",
        int(
            row[
                "seed_schedule_count"
            ]
        ),
    )


    if int(
        row[
            "positive_seed_schedules"
        ]
    ) == int(
        row[
            "seed_schedule_count"
        ]
    ):

        print(
            "  FLAG: effect direction is positive in all five schedules."
        )

    elif int(
        row[
            "negative_seed_schedules"
        ]
    ) == int(
        row[
            "seed_schedule_count"
        ]
    ):

        print(
            "  FLAG: effect direction is negative in all five schedules."
        )

    else:

        print(
            "  FLAG: effect direction changes across seed schedules."
        )


# =====================================================================
# 35. PACKAGE OUTPUT ARTIFACT
# =====================================================================

if FINAL_ZIP.exists():

    FINAL_ZIP.unlink()


with zipfile.ZipFile(
    FINAL_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:

    for path in sorted(
        RESULT_DIR.rglob(
            "*"
        )
    ):

        if path.is_file():

            archive.write(
                path,
                Path(
                    "outputs"
                )
                / path.relative_to(
                    RESULT_DIR
                ),
            )


    # Seed/fold checkpoints are included for full reproducibility.
    for path in sorted(
        CHECKPOINT_DIR.rglob(
            "*.csv"
        )
    ):

        if path.is_file():

            archive.write(
                path,
                Path(
                    "checkpoints"
                )
                / path.relative_to(
                    CHECKPOINT_DIR
                ),
            )


    archive.write(
        protocol_marker,
        Path(
            "protocol"
        )
        / "phase4_protocol_marker.json",
    )


print("\n" + "=" * 105)
print("PHASE 4 COMPLETE")
print("=" * 105)


print(
    "\nFinal artifact:"
)

print(
    FINAL_ZIP
)


print(
    "\nIMPORTANT FILES:"
)

for filename in [
    "seed_schedule.csv",
    "metrics_by_seed_schedule.csv",
    "effects_by_seed_schedule.csv",
    "model_variability_summary.csv",
    "effect_variability_summary.csv",
    "all_selected_epochs.csv",
    "epoch_variability_summary.csv",
    "gate_values_by_seed_and_fold.csv",
    "gate_variability_summary.csv",
    "phase4_protocol.json",
    "phase4_validation.json",
]:

    print(
        " -",
        RESULT_DIR
        / filename,
    )


print(
    "\nUpload the generated ZIP to ChatGPT."
)

print(
    "Do NOT run additional seeds because of the observed result."
)


# =====================================================================
# 36. AUTOMATIC GOOGLE COLAB DOWNLOAD
# =====================================================================

try:

    from google.colab import files

    print(
        "\nStarting automatic ZIP download..."
    )

    files.download(
        str(
            FINAL_ZIP
        )
    )

except Exception as exc:

    print(
        "\nAutomatic download was unavailable:"
    )

    print(
        exc
    )

    print(
        "\nDownload manually from the Files panel:"
    )

    print(
        FINAL_ZIP
    )

IEEE ACCESS R1 — PHASE 4
REPEATED-TRAINING-SEED VARIABILITY

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4

✓ New frozen Phase 4 protocol created.

Cloning frozen research repository...
✓ Frozen repository commit verified:
e5fdb3ceafa08a82789264a8d57fe8f51a8daea1

Installing repository requirements...
✓ Requirements installed.

Frozen seed schedules:
fold            1     2     3     4     5
schedule_id                              
S0            142   242   342   442   542
S1           1142  1242  1342  1442  1542
S2           2142  2242  2342  2442  2542
S3           3142  3242  3342  3442  3542
S4           4142  4242  4342  4442  4542

PREPARING FROZEN ORIGINAL S0
✓ Original S0 Elite predictions loaded.
✓ Original S0 Global predictions loaded.

PHASE 4 — ELITE REPEATED SEEDS

New training jobs remaining: 20

ELITE — reconstructing StatsBomb graphs...


/usr/local/lib/python3.13/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/statsbombpy/api_client.py:27: 

✓ ELITE required frozen graphs verified.

---------------------------------------------------------------------------------------------------------
ELITE FOLD 1
Outer train: 251
Outer test: 166
Inner train: 134
Inner validation: 51
New schedules: ['S1', 'S2', 'S3', 'S4']

...............................................................................................
ELITE | S1 | Fold 1 | Seed 1142
...............................................................................................
  A-MLP epoch: 1
  C-Hybrid positional epoch: 14
  Positional lambda: 0.49049
  C-Hybrid globalmean epoch: 16
  Globalmean lambda: 0.476708


/tmp/ipykernel_504/359501042.py:2802: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  fold_predictions = pd.concat(


✓ CHECKPOINT SAVED: elite S1 fold 1

...............................................................................................
ELITE | S2 | Fold 1 | Seed 2142
...............................................................................................
  A-MLP epoch: 4
  C-Hybrid positional epoch: 43
  Positional lambda: 0.457629
  C-Hybrid globalmean epoch: 7
  Globalmean lambda: 0.48924


...

✓ CHECKPOINT SAVED: global S3 fold 5

...............................................................................................
GLOBAL | S4 | Fold 5 | Seed 4542
...............................................................................................
  A-MLP epoch: 1
  C-Hybrid positional epoch: 5
  Positional lambda: 0.476559
✓ CHECKPOINT SAVED: global S4 fold 5


/tmp/ipykernel_504/359501042.py:2802: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  fold_predictions = pd.concat(


✓ ELITE prediction integrity passed: 5 schedules x 3 models x 454 OOF observations.
✓ GLOBAL prediction integrity passed: 5 schedules x 2 models x 1745 OOF observations.

PHASE 4 — METRICS BY SEED SCHEDULE
cohort seed_schedule               model    n  accuracy  macro_f1  roc_auc
 elite            S0               A-MLP  454  0.698238  0.696220 0.745303
 elite            S0 C-Hybrid-globalmean  454  0.693833  0.687016 0.756891
 elite            S0 C-Hybrid-positional  454  0.731278  0.728240 0.768012
 elite            S1               A-MLP  454  0.696035  0.695941 0.761471
 elite            S1 C-Hybrid-globalmean  454  0.715859  0.714695 0.777931
 elite            S1 C-Hybrid-positional  454  0.722467  0.720711 0.757667
 elite            S2               A-MLP  454  0.715859  0.715626 0.772748
 elite            S2 C-Hybrid-globalmean  454  0.733480  0.731289 0.791925
 elite            S2 C-Hybrid-positional  454  0.676211  0.676197 0.743420
 elite            S3               A-MLP  45

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>